# Depth Estimation Métrico para SLAM — TUM Dataset
**Projeto:** Aprimoramento de SLAM Monocular com Estimação de Profundidade

Este notebook gera depth maps **métricos** usando **Depth Anything V2 Metric (Hypersim)**.


## Modelo
- `dav2_metric_vitl` — DAV2 Metric Large Hypersim (indoor, melhor qualidade)


In [ ]:
# Verificar GPU e montar Drive

import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM total: {total:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
print('Drive montado!')

GPU disponível: True
GPU: Tesla T4
VRAM total: 14.6 GB
Mounted at /content/drive
Drive montado!


In [ ]:
# Instalar dependências

import os

# Clonar repositório DAV2 (inclui metric_depth)
if not os.path.exists('/content/Depth-Anything-V2'):
    !git clone https://github.com/DepthAnything/Depth-Anything-V2 /content/Depth-Anything-V2 -q

# Instalar requirements do metric_depth
!pip install -r /content/Depth-Anything-V2/metric_depth/requirements.txt -q
!pip install -r /content/Depth-Anything-V2/requirements.txt -q

import sys
sys.path.append('/content/Depth-Anything-V2')
sys.path.append('/content/Depth-Anything-V2/metric_depth')

import numpy as np
from PIL import Image
import cv2
import time
import json
import shutil
print('Dependências instaladas!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 8.4 MB/s eta 0:00:00
Dependências instaladas!


In [ ]:
#  CONFIGURAÇÕES 

# Dataset — escolha um:
DATASET = 'fr3_office'  # 'fr1_desk' | 'fr2_xyz' | 'fr3_office'

# Modelo métrico
MODEL_TYPE = 'dav2_metric_vitl'  # nome da pasta de saída
ENCODER    = 'vitl'              # 'vitl' | 'vitb' | 'vits'
MAX_DEPTH  = 20                  # 20m para indoor (Hypersim)

# Pasta de saída no Drive
DRIVE_OUTPUT = f'/content/drive/MyDrive/orbslam_dav2_metric/{DATASET}/{MODEL_TYPE}'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Configurações por dataset
DATASET_CONFIG = {
    'fr1_desk': {
        'url': 'https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz',
        'folder': 'rgbd_dataset_freiburg1_desk',
        'depth_factor': 5000.0  # fator para converter metros -> uint16
    },
    'fr2_xyz': {
        'url': 'https://cvg.cit.tum.de/rgbd/dataset/freiburg2/rgbd_dataset_freiburg2_xyz.tgz',
        'folder': 'rgbd_dataset_freiburg2_xyz',
        'depth_factor': 5000.0
    },
    'fr3_office': {
        'url': 'https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_long_office_household.tgz',
        'folder': 'rgbd_dataset_freiburg3_long_office_household',
        'depth_factor': 5000.0
    }
}

cfg         = DATASET_CONFIG[DATASET]
DEPTH_FACTOR = cfg['depth_factor']  # 5000.0 — padrão TUM
DATASET_DIR  = f'/content/{cfg["folder"]}'

print(f'Dataset      : {DATASET}')
print(f'Modelo       : {MODEL_TYPE} (encoder={ENCODER})')
print(f'MAX_DEPTH    : {MAX_DEPTH}m (indoor Hypersim)')
print(f'DEPTH_FACTOR : {DEPTH_FACTOR} (padrão TUM — sem calibração necessária)')
print(f'Drive output : {DRIVE_OUTPUT}')

Dataset      : fr3_office
Modelo       : dav2_metric_vitl (encoder=vitl)
MAX_DEPTH    : 20m (indoor Hypersim)
DEPTH_FACTOR : 5000.0 (padrão TUM — sem calibração necessária)
Drive output : /content/drive/MyDrive/orbslam_dav2_metric/fr3_office/dav2_metric_vitl


In [ ]:
# Baixar e extrair dataset TUM

if not os.path.exists(DATASET_DIR):
    print(f'Baixando {DATASET}...')
    !wget -q --show-progress {cfg['url']} -O /content/dataset.tgz
    print('Extraindo...')
    !tar -xzf /content/dataset.tgz -C /content/
    !rm /content/dataset.tgz
    print('Dataset pronto!')
else:
    print(f'Dataset já existe: {DATASET_DIR}')

print(f'Imagens RGB   : {len(os.listdir(DATASET_DIR + "/rgb"))}')
print(f'Imagens depth : {len(os.listdir(DATASET_DIR + "/depth"))}')

Baixando fr3_office...
/content/dataset.tg 100%[===================>]   1.38G  19.0MB/s    in 78s     
Extraindo...
Dataset pronto!
Imagens RGB   : 2585
Imagens depth : 2509


In [ ]:
#  Carregar modelo DAV2 MÉTRICO (Hypersim indoor)

import torch
import sys

# IMPORTANTE: usar a versão metric_depth que tem suporte a max_depth
sys.path.insert(0, '/content/Depth-Anything-V2/metric_depth')

# Recarregar o módulo para garantir que usa a versão correta
import importlib
import depth_anything_v2.dpt as dpt_module
importlib.reload(dpt_module)
from depth_anything_v2.dpt import DepthAnythingV2

print(f'Usando: {dpt_module.__file__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64,  'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

weights_url = {
    'vits': 'https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Small/resolve/main/depth_anything_v2_metric_hypersim_vits.pth',
    'vitb': 'https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Base/resolve/main/depth_anything_v2_metric_hypersim_vitb.pth',
    'vitl': 'https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Large/resolve/main/depth_anything_v2_metric_hypersim_vitl.pth',
}

weights_path = f'/content/depth_anything_v2_metric_hypersim_{ENCODER}.pth'
if not os.path.exists(weights_path):
    print(f'Baixando pesos métricos {ENCODER}...')
    !wget -q --show-progress {weights_url[ENCODER]} -O {weights_path}
    print('Download concluído!')
else:
    print(f'Pesos já existem: {weights_path}')

# Instanciar modelo COM max_depth
model = DepthAnythingV2(**{**model_configs[ENCODER], 'max_depth': MAX_DEPTH})
model.load_state_dict(torch.load(weights_path, map_location='cpu'))
model.eval().to(DEVICE)

print(f'DAV2 Metric {ENCODER.upper()} Hypersim carregado!')
print(f'Max depth: {MAX_DEPTH}m (indoor)')

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM usada: {used:.2f} GB / {total:.1f} GB')


Usando: /content/Depth-Anything-V2/metric_depth/depth_anything_v2/dpt.py
Device: cuda
Baixando pesos métricos vitl...
/content/depth_anyt 100%[===================>]   1.25G  20.3MB/s    in 64s     
Download concluído!
DAV2 Metric VITL Hypersim carregado!
Max depth: 20m (indoor)
VRAM usada: 1.25 GB / 14.6 GB


In [ ]:
#  Função de pós-processamento MÉTRICO
# multiplica pelo depth_factor

def postprocess_metric(depth_m, orig_w, orig_h, depth_factor):
    """
    Converte depth métrico (metros, float32) para PNG 16-bit compatível com TUM.
    depth_m: numpy array float32 em METROS (saída direta do modelo métrico)
    depth_factor: 5000.0 (padrão TUM — 1m = 5000 unidades)
    """
    # Redimensionar para resolução original se necessário
    if depth_m.shape != (orig_h, orig_w):
        depth_pil = Image.fromarray(depth_m).resize((orig_w, orig_h), Image.BILINEAR)
        depth_m = np.array(depth_pil, dtype=np.float32)

    # Clip para range válido (0 a max_depth metros)
    depth_m = np.clip(depth_m, 0, MAX_DEPTH)

    # Converter metros → uint16 no formato TUM (1m = depth_factor unidades)
    depth_uint16 = (depth_m * depth_factor).astype(np.uint16)

    return depth_uint16

print('Função postprocess_metric carregada!')
print(f'Fórmula: depth_uint16 = depth_metros * {DEPTH_FACTOR}')
print(f'(sem normalização — modelo já retorna metros reais)')

Função postprocess_metric carregada!
Fórmula: depth_uint16 = depth_metros * 5000.0
(sem normalização — modelo já retorna metros reais)


In [ ]:
#  Inferência em todas as imagens

from tqdm.notebook import tqdm

# Ler lista de imagens
rgb_txt    = os.path.join(DATASET_DIR, 'rgb.txt')
rgb_entries = []
with open(rgb_txt) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) >= 2:
            rgb_entries.append((parts[0], parts[1]))

print(f'Total de frames: {len(rgb_entries)}')

# Pasta de saída
depth_folder = f'depth_{MODEL_TYPE}'
DEPTH_OUT    = os.path.join(DATASET_DIR, depth_folder)
os.makedirs(DEPTH_OUT, exist_ok=True)

associations = []
times        = []
depth_stats  = []  # guardar stats para validação

for timestamp, rel_path in tqdm(rgb_entries, desc=f'{MODEL_TYPE} inference'):
    rgb_path = os.path.join(DATASET_DIR, rel_path)
    if not os.path.exists(rgb_path):
        continue

    t0 = time.time()

    # DAV2 Métrico — usa cv2 BGR — retorna metros
    img_bgr = cv2.imread(rgb_path)
    orig_h, orig_w = img_bgr.shape[:2]
    with torch.no_grad():
        depth_m = model.infer_image(img_bgr)  # HxW em metros (float32)

    times.append(time.time() - t0)

    # Pós-processar — SEM normalização, SEM SCALE
    depth_uint16 = postprocess_metric(depth_m, orig_w, orig_h, DEPTH_FACTOR)

    depth_filename = f'{timestamp}.png'
    Image.fromarray(depth_uint16).save(os.path.join(DEPTH_OUT, depth_filename))

    associations.append(
        f'{timestamp} rgb/{os.path.basename(rel_path)} '
        f'{timestamp} {depth_folder}/{depth_filename}'
    )

    # Guardar stats de alguns frames para validação
    if len(depth_stats) < 10:
        depth_stats.append({
            'ts': timestamp,
            'mean_m': float(depth_m.mean()),
            'max_m':  float(depth_m.max()),
            'min_m':  float(depth_m[depth_m > 0].min()) if (depth_m > 0).any() else 0
        })

avg_ms = np.mean(times) * 1000
print(f'\nInferência concluída!')
print(f'Frames processados : {len(associations)}')
print(f'Latência média     : {avg_ms:.1f} ms/frame')
print(f'FPS médio          : {1000/avg_ms:.1f}')

Total de frames: 2585


dav2_metric_vitl inference:   0%|          | 0/2585 [00:00<?, ?it/s]


Inferência concluída!
Frames processados : 2585
Latência média     : 611.4 ms/frame
FPS médio          : 1.6


In [ ]:
#  Salvar associations e compactar para Drive

# Salvar associations
assoc_filename = f'associations_{MODEL_TYPE}.txt'
assoc_path     = os.path.join(DATASET_DIR, assoc_filename)
with open(assoc_path, 'w') as f:
    f.write('\n'.join(associations))
print(f'Associations salvo: {assoc_path} ({len(associations)} pares)')

# Salvar stats
stats = {
    'modelo':             MODEL_TYPE,
    'encoder':            ENCODER,
    'dataset':            DATASET,
    'tipo':               'metrico_hypersim_indoor',
    'max_depth_m':        MAX_DEPTH,
    'depth_factor':       DEPTH_FACTOR,
    'frames':             len(associations),
    'latencia_media_ms':  float(np.mean(times) * 1000),
    'latencia_std_ms':    float(np.std(times)  * 1000),
    'fps_medio':          float(1000 / (np.mean(times) * 1000)),
    'nota':               'Saida em metros reais — sem calibracao de escala necessaria'
}
stats_path = os.path.join(DRIVE_OUTPUT, f'stats_{DATASET}_{MODEL_TYPE}.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Stats: {json.dumps(stats, indent=2)}')

# Compactar depth para Drive
zip_name = f'depth_{MODEL_TYPE}_{DATASET}'
zip_path = os.path.join(DRIVE_OUTPUT, zip_name)
print(f'\nCompactando {depth_folder}...')
shutil.make_archive(zip_path, 'zip', DATASET_DIR, depth_folder)
print(f'ZIP salvo: {zip_path}.zip')

# Copiar associations para Drive
shutil.copy(assoc_path, os.path.join(DRIVE_OUTPUT, assoc_filename))
print(f'Associations copiado para Drive!')

print('\n=== ARQUIVOS NO DRIVE ===')
!ls -lh {DRIVE_OUTPUT}

Associations salvo: /content/rgbd_dataset_freiburg3_long_office_household/associations_dav2_metric_vitl.txt (2585 pares)
Stats: {
  "modelo": "dav2_metric_vitl",
  "encoder": "vitl",
  "dataset": "fr3_office",
  "tipo": "metrico_hypersim_indoor",
  "max_depth_m": 20,
  "depth_factor": 5000.0,
  "frames": 2585,
  "latencia_media_ms": 611.3905277658016,
  "latencia_std_ms": 25.776653382577408,
  "fps_medio": 1.6356157882495992,
  "nota": "Saida em metros reais \u2014 sem calibracao de escala necessaria"
}

Compactando depth_dav2_metric_vitl...
ZIP salvo: /content/drive/MyDrive/orbslam_dav2_metric/fr3_office/dav2_metric_vitl/depth_dav2_metric_vitl_fr3_office.zip
Associations copiado para Drive!

=== ARQUIVOS NO DRIVE ===
total 675M
-rw------- 1 root root 271K Jun 27 12:52 associations_dav2_metric_vitl.txt
-rw------- 1 root root 675M Jun 27 12:52 depth_dav2_metric_vitl_fr3_office.zip
-rw------- 1 root root  380 Jun 27 12:52 stats_fr3_office_dav2_metric_vitl.json


In [ ]:
# ============================================================
# CÉLULA EXTRA — Regerar depth com correção de escala e salvar separado
# ============================================================

import os
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import shutil
import json

# Fator de correção calculado da análise
SCALE_CORRECTION = 1.34 / 2.35  # ≈ 0.570

# Nome do modelo corrigido
MODEL_TYPE_CORR  = f'{MODEL_TYPE}_scaled'
depth_folder_corr = f'depth_{MODEL_TYPE_CORR}'
DEPTH_OUT_CORR    = os.path.join(DATASET_DIR, depth_folder_corr)
os.makedirs(DEPTH_OUT_CORR, exist_ok=True)

DRIVE_OUTPUT_CORR = f'/content/drive/MyDrive/orbslam_dav2_metric/{DATASET}/{MODEL_TYPE_CORR}'
os.makedirs(DRIVE_OUTPUT_CORR, exist_ok=True)

print(f'Fator de correção : {SCALE_CORRECTION:.4f}')
print(f'Pasta de saída    : {DEPTH_OUT_CORR}')
print(f'Drive output      : {DRIVE_OUTPUT_CORR}')

# Ler arquivos sintéticos já gerados e aplicar correção
sint_files     = sorted(os.listdir(DEPTH_OUT))
associations_corr = []

for fname in tqdm(sint_files, desc='Aplicando correção de escala'):
    sint_path = os.path.join(DEPTH_OUT, fname)
    img = np.array(Image.open(sint_path)).astype(np.float32)

    # Aplicar correção de escala
    img_corr = np.clip(img * SCALE_CORRECTION, 0, 65535).astype(np.uint16)

    # Salvar
    Image.fromarray(img_corr).save(os.path.join(DEPTH_OUT_CORR, fname))

    timestamp = fname.replace('.png', '')
    associations_corr.append(
        f'{timestamp} rgb/{fname} '
        f'{timestamp} {depth_folder_corr}/{fname}'
    )

print(f'\nFrames corrigidos: {len(associations_corr)}')

# Verificar qualidade
print('\n=== DEPTH CORRIGIDO ===')
for fname in sint_files[:5]:
    img = np.array(Image.open(os.path.join(DEPTH_OUT_CORR, fname)))
    print(f'{fname}: mean={img.mean()/DEPTH_FACTOR:.2f}m  max={img.max()/DEPTH_FACTOR:.2f}m')

print('\n=== DEPTH ORIGINAL ===')
for fname in sint_files[:5]:
    img = np.array(Image.open(os.path.join(DEPTH_OUT, fname)))
    print(f'{fname}: mean={img.mean()/DEPTH_FACTOR:.2f}m  max={img.max()/DEPTH_FACTOR:.2f}m')

print('\n=== DEPTH REAL TUM ===')
real_files = sorted(os.listdir(depth_real_dir))
for fname in real_files[:5]:
    img = np.array(Image.open(os.path.join(depth_real_dir, fname)))
    print(f'{fname}: mean={img.mean()/5000:.2f}m  max={img.max()/5000:.2f}m')

# Salvar associations
assoc_filename_corr = f'associations_{MODEL_TYPE_CORR}.txt'
assoc_path_corr     = os.path.join(DATASET_DIR, assoc_filename_corr)
with open(assoc_path_corr, 'w') as f:
    f.write('\n'.join(associations_corr))
print(f'\nAssociations salvo: {assoc_path_corr}')

# Salvar stats
stats_corr = {
    'modelo':            MODEL_TYPE_CORR,
    'modelo_base':       MODEL_TYPE,
    'scale_correction':  SCALE_CORRECTION,
    'dataset':           DATASET,
    'depth_factor':      DEPTH_FACTOR,
    'frames':            len(associations_corr),
    'nota':              f'DAV2 metrico + correcao de escala {SCALE_CORRECTION:.4f}'
}
with open(os.path.join(DRIVE_OUTPUT_CORR, f'stats_{DATASET}_{MODEL_TYPE_CORR}.json'), 'w') as f:
    json.dump(stats_corr, f, indent=2)

# Compactar e enviar para Drive
zip_name = f'depth_{MODEL_TYPE_CORR}_{DATASET}'
zip_path = os.path.join(DRIVE_OUTPUT_CORR, zip_name)
print(f'\nCompactando...')
shutil.make_archive(zip_path, 'zip', DATASET_DIR, depth_folder_corr)
shutil.copy(assoc_path_corr, os.path.join(DRIVE_OUTPUT_CORR, assoc_filename_corr))
print(f'ZIP salvo: {zip_path}.zip')
print(f'Associations copiado para Drive!')

print('\n=== ARQUIVOS NO DRIVE (corrigido) ===')
!ls -lh {DRIVE_OUTPUT_CORR}

Fator de correção : 0.5702
Pasta de saída    : /content/rgbd_dataset_freiburg3_long_office_household/depth_dav2_metric_vitl_scaled
Drive output      : /content/drive/MyDrive/orbslam_dav2_metric/fr3_office/dav2_metric_vitl_scaled


Aplicando correção de escala:   0%|          | 0/2585 [00:00<?, ?it/s]


Frames corrigidos: 2585

=== DEPTH CORRIGIDO ===
1341847980.722988.png: mean=1.71m  max=7.15m
1341847980.754743.png: mean=1.69m  max=7.17m
1341847980.786856.png: mean=1.69m  max=7.19m
1341847980.822978.png: mean=1.70m  max=6.46m
1341847980.854676.png: mean=1.71m  max=6.75m

=== DEPTH ORIGINAL ===
1341847980.722988.png: mean=3.00m  max=12.53m
1341847980.754743.png: mean=2.96m  max=12.58m
1341847980.786856.png: mean=2.96m  max=12.61m
1341847980.822978.png: mean=2.98m  max=11.33m
1341847980.854676.png: mean=2.99m  max=11.84m

=== DEPTH REAL TUM ===
1341847980.723020.png: mean=2.00m  max=9.33m
1341847980.754755.png: mean=2.08m  max=9.87m
1341847980.786879.png: mean=2.07m  max=9.87m
1341847980.822989.png: mean=2.07m  max=9.33m
1341847980.854690.png: mean=2.05m  max=9.59m

Associations salvo: /content/rgbd_dataset_freiburg3_long_office_household/associations_dav2_metric_vitl_scaled.txt

Compactando...
ZIP salvo: /content/drive/MyDrive/orbslam_dav2_metric/fr3_office/dav2_metric_vitl_scaled/d